# 🛰️ CloudVision-RS — Phase 2: Cloud Mask Segmentation Training

**Model**: Attention U-Net (ResNet34 encoder + SCSE decoder attention)  
**Task**: Predict binary cloud mask from 256×256 RGB satellite patches  
**Loss**: Combined BCE + Dice  
**Framework**: PyTorch Lightning + W&B logging  

---

## ✅ Before You Start — Checklist

| Step | Action |
|------|--------|
| 1 | `Runtime → Change runtime type → GPU (T4 or A100)` |
| 2 | Upload your project folder to Google Drive |
| 3 | Update `PROJECT_ROOT` in **Step 1** with your Drive path |
| 4 | Run all cells **top to bottom** — do not skip cells |

---

> ⚠️ **Session Management**: Free Colab disconnects after ~90 min idle.  
> Checkpoints are saved to Google Drive after every epoch, so training **resumes automatically** from the last saved checkpoint if the session drops.

## Step 1 — Mount Google Drive & Set Project Root

Mount your Drive so Colab can access your project files and save checkpoints.

**What to run:**
```python
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/CloudRemoval_Project'  # ← UPDATE THIS
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))
```

✏️ **Change `PROJECT_ROOT`** to match the actual path inside your Google Drive.

## Step 2 — Install Dependencies

Install packages not already in Colab's environment.

**What to run:**
```python
%%capture
!pip install pytorch-lightning segmentation-models-pytorch \
             torchmetrics wandb rich --quiet
```

- `pytorch-lightning` — training loop management  
- `segmentation-models-pytorch` — provides the AttentionUNet with ResNet34 encoder  
- `torchmetrics` — IoU and Dice computed correctly across full epoch  
- `wandb` — experiment tracking (loss curves, metrics, images)  
- `rich` — pretty progress bar in the terminal

## Step 3 — Verify GPU & Environment

Confirm that Colab assigned a GPU and that all imports work.

**What to run:**
```python
import torch
import pytorch_lightning as pl
import segmentation_models_pytorch as smp
import torchmetrics
import wandb

print(f'PyTorch     : {torch.__version__}')
print(f'Lightning   : {pl.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
```

> 💡 If `CUDA: False`, go to `Runtime → Change runtime type → GPU` and reconnect.

## Step 4 — Load Config

Load `configs/seg_config.yaml` and apply Colab-specific overrides.

**What to run:**
```python
import yaml

with open('configs/seg_config.yaml') as f:
    config = yaml.safe_load(f)['segmentation']

# Colab overrides (these override the YAML values for this session)
config['seg_batch_size'] = 16   # T4: 16 | A100: 32
config['num_workers']    = 4

print('Config loaded successfully:')
for k, v in config.items():
    print(f'  {k}: {v}')
```

## Step 5 — Model Sanity Check

Instantiate the model and run a dummy forward pass to confirm shapes are correct before training.

**What to run:**
```python
import sys
sys.path.insert(0, '.')

from src.models.segmentation import AttentionUNet
from src.losses.seg_loss import SegmentationLoss

model = AttentionUNet(config)

dummy_input = torch.zeros(2, 3, 256, 256)
dummy_mask  = torch.randint(0, 2, (2, 1, 256, 256)).float()

output = model(dummy_input)
print(f'Input  shape : {dummy_input.shape}')
print(f'Output shape : {output.shape}')   # expect [2, 1, 256, 256]

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

criterion = SegmentationLoss(bce_weight=0.5, dice_weight=0.5)
losses = criterion(output, dummy_mask)
print(f'Loss dict  : {losses}')
print('✓ Model and loss are working correctly')
```

> **Expected output shape**: `[2, 1, 256, 256]`  
> **Expected parameter count**: ~24.4M (ResNet34 encoder + U-Net decoder)

## Step 6 — Weights & Biases Login

Authenticate with W&B to enable experiment tracking.

**What to run:**
```python
import wandb
wandb.login()   # paste your API key when prompted
                # find it at: https://wandb.ai/authorize
```

> 💡 You only need to do this once per Colab session.  
> Your API key is saved in the runtime for the duration of the session.  
> To skip W&B, change `logger` in `train()` to `CSVLogger` in `train_segmentation.py`.

## Step 7 — Create Output Directories

Create all necessary folders before training starts.

**What to run:**
```python
import os

dirs = [
    config['checkpoint_dir'],
    config['drive_checkpoint_dir'],
    config['log_dir'],
    'outputs/predictions',
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f'Created: {d}')

print('✓ All directories ready')
```

## Step 8 — Run Training

This cell launches the full training pipeline using `train_segmentation.py`.  
Training saves checkpoints to Google Drive after every epoch.

**What to run:**
```python
from src.training.train_segmentation import train

trainer = train(config, is_colab=True)
```

**What happens during training:**
| Epoch | Action |
|-------|--------|
| Every epoch | `last.ckpt` saved to Drive |
| When val IoU improves | `seg-best-*.ckpt` saved to Drive |
| After `patience=12` epochs with no improvement | Training stops early |
| After training | Test set evaluation runs automatically |

**Expected metrics (RICE2 dataset):**
| Metric | Expected range |
|--------|---------------|
| val/iou | 0.82 – 0.91 |
| val/dice | 0.90 – 0.95 |

> ⏱️ **Training time estimate**: ~3 min/epoch on T4 with batch_size=16  
> Full 60 epochs ≈ 3 hours | With early stopping ≈ 1–1.5 hours

## Step 9 — (Optional) Resume After Disconnect

If your Colab session was disconnected, re-run **Steps 1–7** first, then run this cell.  
The trainer will automatically detect `last.ckpt` in your Drive and resume from it.

**What to run:**
```python
from src.training.train_segmentation import train

# is_colab=True tells train() to look in drive_checkpoint_dir for last.ckpt
trainer = train(config, is_colab=True)
```

> 💡 No extra code needed — `train()` already handles resume automatically.  
> It checks for `last.ckpt` in the checkpoint directory at the start.

## Step 10 — Visualise Predictions

After training, visually verify that the model is correctly detecting clouds.

**What to run:**
```python
import matplotlib.pyplot as plt
import numpy as np
from src.training.train_segmentation import SegmentationModule, create_dataloaders

# Load best checkpoint
best_ckpt_path = config['drive_checkpoint_dir'] + '/seg-best-*.ckpt'
import glob
ckpts = sorted(glob.glob(best_ckpt_path))
print('Best checkpoint:', ckpts[-1])

model = SegmentationModule.load_from_checkpoint(ckpts[-1], config=config)
model.eval().cuda()

# Get a batch from the test set
loaders = create_dataloaders(
    patches_dir=config['patches_dir'],
    mode='segmentation',
    config_path='configs/data_config.yaml'
)
batch = next(iter(loaders['test']))
images = batch['image'].cuda()
masks  = batch['mask']

with torch.no_grad():
    logits = model(images)
    preds  = (torch.sigmoid(logits) > 0.5).float().cpu()

# Plot
n = min(4, images.shape[0])
fig, axes = plt.subplots(3, n, figsize=(4*n, 12))
for i in range(n):
    img  = images[i].cpu().permute(1, 2, 0).numpy()
    gt   = masks[i, 0].numpy()
    pred = preds[i, 0].numpy()
    axes[0, i].imshow(np.clip(img, 0, 1)); axes[0, i].set_title('Input'); axes[0, i].axis('off')
    axes[1, i].imshow(gt,   cmap='gray');  axes[1, i].set_title('GT Mask'); axes[1, i].axis('off')
    axes[2, i].imshow(pred, cmap='gray');  axes[2, i].set_title('Prediction'); axes[2, i].axis('off')

plt.suptitle('Row 1: Input  |  Row 2: Ground Truth  |  Row 3: Predicted Mask', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/predictions/test_predictions.png', dpi=120)
plt.show()
print('Saved → outputs/predictions/test_predictions.png')
```

## Step 11 — Plot Training Curves

Inspect loss and metric progression across epochs.

**What to run:**
```python
# Option A: View in W&B dashboard (recommended)
# Go to https://wandb.ai → your project → run: seg-resnet34-attn-unet

# Option B: Plot from CSV logs
import pandas as pd
import glob
import matplotlib.pyplot as plt

log_files = glob.glob('outputs/logs/**/metrics.csv', recursive=True)
if not log_files:
    print('No CSV logs found — check outputs/logs/')
else:
    df = pd.read_csv(sorted(log_files)[-1])
    print('Available columns:', df.columns.tolist())

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for col, ax, title in [
        ('train/loss',  axes[0], 'Train Loss'),
        ('val/iou',     axes[1], 'Val IoU'),
        ('val/dice',    axes[2], 'Val Dice'),
    ]:
        sub = df.dropna(subset=[col])
        if not sub.empty:
            ax.plot(sub['epoch'], sub[col], marker='o', linewidth=2)
        ax.set_title(title); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('outputs/training_curves.png', dpi=120)
    plt.show()
```

---

## ✅ Phase 2 Complete — Summary

| Component | File | Status |
|-----------|------|--------|
| Config | `configs/seg_config.yaml` | ✓ |
| Model | `src/models/segmentation.py` | ✓ |
| Loss | `src/losses/seg_loss.py` | ✓ |
| Training | `src/training/train_segmentation.py` | ✓ |
| Notebook | `notebooks/02_train_segmentation_colab.ipynb` | ✓ |

### Next Steps
- 📊 Review W&B dashboard for loss curves and metric trends
- 🔍 Inspect predictions in `outputs/predictions/`
- 🔧 Tune `dice_weight` to `0.7` if cloud coverage is sparse (<15%)
- ➡️ **Phase 3**: `03_train_inpainting_colab.ipynb` — GAN-based cloud removal using the masks produced by this model